# Thesaurus AI: LLM Fine-Tuning Project

This notebook demonstrates how to fine-tune a Large Language Model (LLM) to create a thesaurus application. It serves as an educational resource to understand the principles and methods of LLM fine-tuning.

## What You'll Learn

- How to prepare data for fine-tuning a thesaurus LLM
- How to use Parameter-Efficient Fine-Tuning (PEFT) with LoRA
- How to evaluate and use your fine-tuned model
- How to build a simple thesaurus application

## Prerequisites

- Basic understanding of Python
- Familiarity with deep learning concepts
- A GPU-enabled environment (recommended)

Let's get started!

## 1. Setup and Installation

First, let's install the required libraries. We'll need:
- `transformers`: For working with pre-trained models
- `peft`: For Parameter-Efficient Fine-Tuning
- `datasets`: For handling datasets
- `torch`: For deep learning operations
- `nltk`: For accessing WordNet (our thesaurus data source)
- Other utility libraries

In [ ]:
# Install required packages
!pip install -q transformers==4.30.2 datasets==2.13.1 peft==0.4.0 torch==2.0.1 accelerate==0.20.3 bitsandbytes==0.40.2 tqdm==4.65.0 nltk==3.8.1 pandas==2.0.3 matplotlib==3.7.2 scikit-learn==1.3.0

In [ ]:
# Import required libraries
import os
import json
import torch
import nltk
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model, TaskType, PeftModel

# Check if GPU is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 2. Understanding LLM Fine-Tuning

### What is LLM Fine-Tuning?

Fine-tuning is the process of taking a pre-trained language model and further training it on a specific dataset to adapt it for a particular task. In our case, we're fine-tuning a model to become a thesaurus.

### Why Use Parameter-Efficient Fine-Tuning (PEFT)?

Traditional fine-tuning updates all parameters of a model, which can be:
- Computationally expensive
- Memory-intensive
- Prone to catastrophic forgetting

PEFT methods like LoRA (Low-Rank Adaptation) allow us to fine-tune large models by adding a small number of trainable parameters while keeping most of the original model frozen.

### How LoRA Works

LoRA works by adding low-rank matrices to the attention layers of the transformer model:

1. For each weight matrix W in the model, LoRA adds a decomposition BA where B and A are low-rank matrices
2. During forward pass: h = Wx + BAx
3. Only B and A are trained, while W remains frozen
4. This drastically reduces the number of trainable parameters

Let's visualize this concept:

In [ ]:
# Create a simple visualization of LoRA
plt.figure(figsize=(10, 6))

# Original weight matrix (frozen)
plt.subplot(1, 2, 1)
plt.title("Traditional Fine-Tuning")
plt.imshow(torch.ones(10, 10), cmap='Blues')
plt.xlabel("All parameters are updated")
plt.colorbar(label="Trainable")

# LoRA decomposition
plt.subplot(1, 2, 2)
plt.title("LoRA Fine-Tuning")
lora_vis = torch.zeros(10, 10)
lora_vis[:, :2] = 1  # Low-rank matrices
lora_vis[:2, :] = 1  # Low-rank matrices
plt.imshow(lora_vis, cmap='Blues')
plt.xlabel("Only low-rank matrices are updated")
plt.colorbar(label="Trainable")

plt.tight_layout()
plt.show()

## 3. Data Preparation

For our thesaurus application, we need data that includes words and their synonyms, antonyms, and related terms. We'll use WordNet from the NLTK library as our data source.

Let's create a dataset suitable for fine-tuning our LLM:

In [ ]:
# Import our data preparation utilities
from data_preparation import generate_thesaurus_data, prepare_training_data

# Generate thesaurus data (this may take a few minutes)
print("Generating thesaurus data...")
thesaurus_data = generate_thesaurus_data(num_words=500)  # Using a smaller dataset for demonstration

# Prepare training data in the instruction format
print("Preparing training data...")
training_examples = prepare_training_data(thesaurus_data)

# Display a few examples
print("\nSample training examples:")
for i, example in enumerate(training_examples[:3]):
    print(f"\nExample {i+1}:")
    print(f"Instruction: {example['instruction']}")
    print(f"Input: {example['input']}")
    print(f"Output: {example['output']}")

### Converting to HuggingFace Dataset

Now, let's convert our training examples to a format suitable for the Transformers library:

In [ ]:
# Convert to HuggingFace Dataset
def format_instruction(example):
    """Format the example as an instruction."""
    if example["input"]:
        return f"### Instruction: {example['instruction']}\n\n### Input: {example['input']}\n\n### Response: {example['output']}"
    else:
        return f"### Instruction: {example['instruction']}\n\n### Response: {example['output']}"

# Create the dataset
dataset_dict = {"text": [format_instruction(ex) for ex in training_examples]}
dataset = Dataset.from_dict(dataset_dict)

# Split into training and validation sets
dataset = dataset.train_test_split(test_size=0.1)
train_dataset = dataset["train"]
eval_dataset = dataset["test"]

print(f"Training examples: {len(train_dataset)}")
print(f"Validation examples: {len(eval_dataset)}")

# Display a formatted example
print("\nFormatted example:")
print(train_dataset[0]["text"])

## 4. Model Selection and Initialization

For this project, we'll use GPT-2 as our base model. It's a good choice for educational purposes because:
- It's relatively small and can be fine-tuned on modest hardware
- It has good language understanding capabilities
- It's well-supported by the Transformers library

Let's load the model and tokenizer:

In [ ]:
# Load the base model and tokenizer
model_name = "gpt2"  # We can also use "gpt2-medium" if more compute is available
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token  # GPT-2 doesn't have a pad token by default

# Load the model
model = AutoModelForCausalLM.from_pretrained(model_name)

# Print model size
param_count = sum(p.numel() for p in model.parameters())
print(f"Model: {model_name}")
print(f"Parameter count: {param_count:,}")

### Tokenizing the Dataset

Now, let's tokenize our dataset for training:

In [ ]:
# Define the tokenization function
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=512)

# Tokenize the datasets
tokenized_train_dataset = train_dataset.map(tokenize_function, batched=True)
tokenized_eval_dataset = eval_dataset.map(tokenize_function, batched=True)

# Set the format for PyTorch
tokenized_train_dataset = tokenized_train_dataset.remove_columns(["text"])
tokenized_train_dataset = tokenized_train_dataset.rename_column("input_ids", "labels")
tokenized_train_dataset.set_format("torch")

tokenized_eval_dataset = tokenized_eval_dataset.remove_columns(["text"])
tokenized_eval_dataset = tokenized_eval_dataset.rename_column("input_ids", "labels")
tokenized_eval_dataset.set_format("torch")

print("Datasets tokenized and formatted for training")

## 5. Applying LoRA for Fine-Tuning

Now, let's apply LoRA to our model for parameter-efficient fine-tuning:

In [ ]:
# Configure LoRA
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,                     # Rank of the update matrices
    lora_alpha=32,           # Parameter for scaling
    lora_dropout=0.1,        # Dropout probability for LoRA layers
    bias="none",             # Don't add bias parameters
    target_modules=["c_attn", "c_proj"]  # Attention modules to apply LoRA to
)

# Apply LoRA to the model
peft_model = get_peft_model(model, lora_config)

# Print trainable parameters
trainable_params = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in peft_model.parameters())
print(f"Trainable parameters: {trainable_params:,} ({trainable_params/total_params:.2%} of total)")

# Print the model architecture
print("\nModel architecture with LoRA:")
print(peft_model)

## 6. Training the Model

Now, let's set up the training configuration and train our model:

In [ ]:
# Define training arguments
training_args = TrainingArguments(
    output_dir="./thesaurus-model",
    overwrite_output_dir=True,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    eval_steps=500,
    save_steps=500,
    warmup_steps=100,
    logging_steps=100,
    evaluation_strategy="steps",
    load_best_model_at_end=True,
    learning_rate=1e-4,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),  # Use mixed precision if GPU is available
    report_to="none"
)

# Create data collator
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False  # We're not doing masked language modeling
)

# Create trainer
trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_eval_dataset,
    data_collator=data_collator,
)

# Train the model
print("Starting training...")
trainer.train()

### Saving the Model

Let's save our fine-tuned model:

In [ ]:
# Save the LoRA adapter
peft_model.save_pretrained("./thesaurus-model-lora")

# Save the tokenizer
tokenizer.save_pretrained("./thesaurus-model-lora")

print("Model and tokenizer saved to ./thesaurus-model-lora")

## 7. Evaluating the Model

Let's evaluate our fine-tuned model on some test examples:

In [ ]:
# Load the fine-tuned model
base_model = AutoModelForCausalLM.from_pretrained(model_name)
fine_tuned_model = PeftModel.from_pretrained(base_model, "./thesaurus-model-lora")
fine_tuned_model = fine_tuned_model.to(device)
fine_tuned_model.eval()

# Function to generate text
def generate_text(prompt, max_length=100):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    outputs = fine_tuned_model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_length=max_length,
        temperature=0.7,
        top_p=0.9,
        do_sample=True
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Test with some examples
test_prompts = [
    "### Instruction: List synonyms for the word 'happy'.\n\n### Response:",
    "### Instruction: What are the antonyms of 'good'?\n\n### Response:",
    "### Instruction: What are more general terms (hypernyms) for 'dog'?\n\n### Response:",
    "### Instruction: Is 'joyful' a synonym of 'happy'?\n\n### Response:"
]

print("Testing the fine-tuned model:")
for prompt in test_prompts:
    print("\nPrompt:")
    print(prompt)
    print("\nGenerated response:")
    response = generate_text(prompt)
    print(response)

## 8. Building a Thesaurus Application

Now, let's use our fine-tuned model to build a simple thesaurus application:

In [ ]:
# Import the ThesaurusLLM class
from thesaurus_utils import ThesaurusLLM

# Initialize the thesaurus
thesaurus = ThesaurusLLM("./thesaurus-model-lora", device=device)

# Function to display thesaurus results
def display_thesaurus_results(word):
    print(f"\n{'='*50}")
    print(f"Thesaurus results for '{word}':\n")
    
    # Get synonyms
    synonyms = thesaurus.get_synonyms(word)
    print(f"Synonyms: {', '.join(synonyms)}\n")
    
    # Get antonyms
    antonyms = thesaurus.get_antonyms(word)
    print(f"Antonyms: {', '.join(antonyms)}\n")
    
    # Get hypernyms (more general terms)
    hypernyms = thesaurus.get_related_terms(word, relation_type="hypernyms")
    print(f"More general terms: {', '.join(hypernyms)}\n")
    
    # Get hyponyms (more specific terms)
    hyponyms = thesaurus.get_related_terms(word, relation_type="hyponyms")
    print(f"More specific terms: {', '.join(hyponyms)}")
    print(f"{'='*50}")

# Test the thesaurus application with some words
test_words = ["happy", "book", "run"]
for word in test_words:
    display_thesaurus_results(word)

## 9. Interactive "Need Help" Section

Let's create an interactive section where users can ask questions about the thesaurus or LLM fine-tuning:

In [ ]:
def ask_question(question):
    """Function to ask a question to the fine-tuned model."""
    prompt = f"### Instruction: {question}\n\n### Response:"
    response = generate_text(prompt, max_length=200)
    
    # Extract just the response part
    response = response.split("### Response:")[-1].strip()
    return response

# Example questions
example_questions = [
    "What is a synonym?",
    "How does LoRA fine-tuning work?",
    "What's the difference between a hypernym and a hyponym?"
]

print("Need Help? Ask a question about thesaurus terms or LLM fine-tuning!")
print("Example questions:")
for i, q in enumerate(example_questions):
    print(f"{i+1}. {q}")

# Interactive question answering
user_question = input("\nEnter your question (or type 'exit' to quit): ")
while user_question.lower() != "exit":
    if user_question:
        print("\nAnswer:")
        print(ask_question(user_question))
    user_question = input("\nEnter your question (or type 'exit' to quit): ")

## 10. Conclusion and Next Steps

Congratulations! You've successfully:

1. Prepared thesaurus data for fine-tuning
2. Applied LoRA for parameter-efficient fine-tuning
3. Fine-tuned a GPT-2 model to act as a thesaurus
4. Built a simple thesaurus application
5. Created an interactive help section

### Next Steps

To further improve your thesaurus LLM, you could:

1. **Use a larger base model**: Try GPT-2 Medium or Large for better performance
2. **Expand the training data**: Include more words and relationships
3. **Fine-tune for longer**: Increase the number of training epochs
4. **Try different PEFT methods**: Experiment with other techniques like Prefix Tuning or P-Tuning
5. **Build a web interface**: Create a user-friendly web application for your thesaurus

### Key Takeaways

- LLM fine-tuning can adapt pre-trained models for specific tasks
- Parameter-Efficient Fine-Tuning (PEFT) makes fine-tuning more accessible
- LoRA is an effective technique for fine-tuning with limited resources
- Fine-tuned models can be used to build practical applications

Thank you for exploring LLM fine-tuning with this thesaurus project!